# Natural Language Processing with Classification and Vector Spaces

Этот раздел закладывает фундамент классического NLP. Именно здесь появляется базовая логика почти всех ранних NLP-pipeline:

**текст → предобработка → векторное представление → модель → оценка качества**.

В субтитрах курса эта логика проходит через весь первый блок: сначала текст очищается и превращается в признаки, затем на этих признаках обучается классификатор, после чего качество проверяется на отложенной выборке. Для logistic regression курс прямо формулирует цепочку: process data → train model → test accuracy; для представления текста как вектора вводятся vocabulary, sparse representation и feature extraction; для Naive Bayes — вероятностная интерпретация слов и классов, сглаживание и log-likelihood.

---

## 1. Место раздела в общей карте знаний NLP

В общей иерархии знаний этот раздел расположен так:

```text
NLP
├── Text processing
│   ├── tokenization
│   ├── normalization
│   ├── stop words
│   ├── stemming / lemmatization
│   └── regex-based cleaning
├── Text representation
│   ├── Bag of Words
│   ├── Document-Term Matrix
│   ├── TF
│   ├── TF-IDF
│   ├── co-occurrence representations
│   └── vector space models
├── Classical classification
│   ├── Logistic Regression
│   ├── Naive Bayes
│   ├── k-NN
│   └── linear decision boundaries
├── Similarity and retrieval
│   ├── cosine similarity
│   ├── Euclidean distance
│   ├── nearest neighbors
│   └── information retrieval
└── Dimensionality reduction
    ├── SVD
    ├── PCA
    └── LSA
```

### Что здесь фундаментально
Фундаментальными темами являются:

- preprocessing;
- vocabulary и token representation;
- Bag of Words / TF-IDF;
- вероятностная и линейная постановка классификации;
- similarity в векторном пространстве.

### Что здесь производно
Производными темами являются:

- конкретные модели классификации: Logistic Regression, Naive Bayes, k-NN;
- retrieval через nearest neighbors;
- снижение размерности PCA/SVD/LSA.

Именно этот раздел создаёт язык, на котором потом объясняются embeddings, sequence models и transformers. Без понимания того, что такое вектор текста, признак, расстояние, вероятность класса и ошибка модели, последующие разделы будут восприниматься как набор архитектур без основания.

---

## 2. Онтология ключевых терминов раздела

### Tokenization
- **тип:** preprocessing method  
- **определение:** разбиение текста на токены: слова, подслова, знаки препинания или другие единицы.
- **связи:** vocabulary, stop words, stemming, Bag of Words.
- **роль:** первый шаг почти любого NLP pipeline.

### Vocabulary
- **тип:** data representation object  
- **определение:** множество уникальных токенов корпуса.
- **связи:** tokenization, Bag of Words, sparse vectors.
- **роль:** определяет пространство признаков.

### Stop Words
- **тип:** preprocessing heuristic  
- **определение:** высокочастотные слова с малой полезностью для конкретной задачи.
- **связи:** preprocessing, sentiment analysis, information loss.
- **роль:** уменьшают размер пространства признаков, но иногда удаляют важные слова вроде *not*.

### Stemming
- **тип:** preprocessing method  
- **определение:** приведение словоформ к грубой основе.
- **связи:** normalization, vocabulary reduction.
- **роль:** уменьшает разреженность и размер словаря.

### Bag of Words
- **тип:** text representation model  
- **определение:** представление текста вектором частот или индикаторов слов без учёта порядка.
- **связи:** vocabulary, document-term matrix, TF-IDF, Naive Bayes, Logistic Regression.
- **роль:** базовый способ перевести текст в числовую форму.

### Sparse Representation
- **тип:** vector property  
- **определение:** представление, в котором большинство координат равны нулю.
- **связи:** Bag of Words, high-dimensional data, efficiency.
- **роль:** типичный формат классического NLP.

### TF-IDF
- **тип:** weighted vector representation  
- **определение:** вес слова, пропорциональный частоте в документе и обратно пропорциональный распространённости в коллекции.
- **связи:** Bag of Words, retrieval, linear models.
- **роль:** усиливает информативные слова и подавляет общие.

### Logistic Regression
- **тип:** discriminative classification model  
- **определение:** линейная модель бинарной классификации, предсказывающая вероятность класса через sigmoid.
- **связи:** features, gradient descent, cross-entropy loss.
- **роль:** сильный baseline для текстовой классификации.

### Naive Bayes
- **тип:** probabilistic generative classifier  
- **определение:** модель, вычисляющая вероятность класса по вероятностям слов при допущении условной независимости признаков.
- **связи:** Bayes rule, conditional probability, Laplace smoothing, log-likelihood.
- **роль:** быстрый и интерпретируемый baseline для текста.

### Cosine Similarity
- **тип:** similarity measure  
- **определение:** мера близости между векторами по углу между ними.
- **связи:** vector spaces, k-NN, search.
- **роль:** стандартная метрика для сравнения текстов и словарных векторов.

### PCA / SVD / LSA
- **тип:** dimensionality reduction methods  
- **определение:** методы проецирования данных в пространство меньшей размерности с сохранением важной структуры.
- **связи:** vector spaces, latent semantics, visualization.
- **роль:** помогают выявлять скрытые факторы и уменьшать шум.

---

# 3. Text Preprocessing

## 3.1. Интуитивная идея

Сырые тексты почти никогда не готовы для модели. В них есть:

- разные регистры;
- URL, упоминания, эмодзи, пунктуация;
- морфологические варианты слов;
- служебные слова;
- шум.

Предобработка нужна затем, чтобы сделать тексты **сопоставимыми**, уменьшить размер словаря и выделить сигнал, полезный для задачи. В субтитрах курса preprocessing для sentiment analysis включает lowercasing, удаление punctuation, URLs и handles, удаление stop words, stemming и tokenization. Одновременно курс подчёркивает, что удаление не должно быть механическим: например, punctuation и слово *not* могут нести критически важный смысл.

## 3.2. Формальная постановка

Пусть документ — это последовательность символов:

$$
d = (c_1, c_2, \dots, c_n)
$$

После tokenization и normalization он преобразуется в последовательность токенов:

$$
d' = (t_1, t_2, \dots, t_m)
$$

Затем применяется отображение предобработки:

$$
\phi_{\text{prep}}: d \mapsto \tilde d
$$

где $\tilde d$ — очищенный и нормализованный набор токенов.

Обычно $\phi_{\text{prep}}$ состоит из композиции:

$$
\phi_{\text{prep}} = \phi_{\text{tokenize}} \circ \phi_{\text{normalize}} \circ \phi_{\text{filter}} \circ \phi_{\text{stem/lemma}}
$$

## 3.3. Алгоритм

1. Разбить текст на токены.
2. Привести к нижнему регистру.
3. Удалить URL, user handles и технический мусор.
4. Решить, сохранять ли punctuation.
5. Удалить stop words, если это не уничтожает смысл.
6. Выполнить stemming или lemmatization.
7. Вернуть нормализованный список токенов.

## 3.4. Псевдокод

```text
function preprocess(text):
    text = lowercase(text)
    text = remove_urls(text)
    text = remove_handles(text)
    tokens = tokenize(text)
    tokens = remove_punctuation(tokens)      # если уместно
    tokens = remove_stopwords(tokens)        # осторожно для negation
    tokens = stem_or_lemmatize(tokens)
    return tokens
```

## 3.5. Пример

Исходный текст:

```text
@user Tuning GREAT AI model!!! https://site.com
```

После preprocessing:

```text
["tun", "great", "ai", "model"]
```

Это соответствует разбору из курса: handles и URLs удаляются, регистр нормализуется, слова приводятся к основе, а размер vocabulary уменьшается.

## 3.6. Применение в NLP

- sentiment analysis;
- spam detection;
- authorship attribution;
- information retrieval;
- topic classification.

## 3.7. Ограничения

Главная ошибка новичков — считать preprocessing всегда полезным. Это неверно.

Примеры проблем:

- удаление *not* ломает полярность;
- удаление punctuation уничтожает эмотиконы и эмоциональные сигналы;
- stemming может искажать слова;
- чрезмерная очистка уничтожает стиль и прагматику текста.

Курс специально показывает, что фраза типа *This is not good* после удаления stop words может превратиться в набор позитивных слов, а sad-face punctuation может быть критична для sentiment.

## 3.8. Исторический контекст

В классическом NLP preprocessing был почти обязательным и часто очень агрессивным, потому что признаки строились вручную и модели плохо справлялись с шумом. В современных трансформерах часть этих процедур либо ослабляется, либо переносится на уровень tokenizer/subword models.

## 3.9. Связь с другими методами

Preprocessing напрямую влияет на:

- размер vocabulary;
- разреженность Bag of Words;
- устойчивость Naive Bayes;
- качество TF-IDF;
- эффективность линейных моделей.

---

# 4. Bag of Words и Document-Term Matrix

**Bag of Words** (BoW) — это один из самых базовых способов представить текст в виде числового вектора для машинного обучения. BoW игнорирует порядок слов и рассматривает текст просто как набор слов (мешок слов). BoW считает сколько раз каждое слово встретиловь в тексте, таким образом превращая текст в вектор признаков, где: каждое слово = отдельная "фича", значение = важность/частота слова.

## 4.1. Интуитивная идея

Чтобы обучить модель, текст надо превратить в числа. Самый прямой способ: создать словарь всех уникальных слов корпуса и для каждого документа отметить, какие слова в нём есть и сколько раз они встретились.

Это и есть идея **Bag of Words**: документ рассматривается как мешок слов без порядка.

Курс описывает именно такую схему: строится vocabulary, затем для каждого слова из vocabulary проверяется, встречается ли оно в tweet, после чего формируется вектор признаков. Такой вектор почти всегда разреженный.

## 4.2. Формальная постановка

Пусть словарь:

$$
V = \{w_1, w_2, \dots, w_{|V|}\}
$$

Документ $d$ представляется вектором:

$$
x(d) = (x_1, x_2, \dots, x_{|V|})
$$

где:

- в бинарной модели  
$$
x_j = \mathbb{1}[w_j \in d]
$$

- в count-based модели  
$$
x_j = \text{count}(w_j, d)
$$

Если у нас $N$ документов, получаем матрицу документ-признак:

$$
X \in \mathbb{R}^{N \times |V|}
$$

Эта матрица называется **Document-Term Matrix**.

## 4.3. Алгоритм

1. Собрать vocabulary.
2. Зафиксировать индекс каждого слова.
3. Для каждого документа:
   - пройти по токенам;
   - посчитать occurrences;
   - записать их в соответствующие координаты вектора.
4. Собрать все векторы в матрицу.

## 4.4. Псевдокод

```text
function build_vocabulary(corpus):
    vocab = unique_tokens(corpus)
    return {word: idx for idx, word in enumerate(vocab)}

function vectorize_bow(document, vocab):
    x = zero_vector(len(vocab))
    for token in document:
        if token in vocab:
            x[vocab[token]] += 1
    return x
```

## 4.5. Пример

Пусть vocabulary:

```text
["i", "am", "happy", "sad", "learning", "nlp"]
```

Документ:

```text
["i", "am", "happy", "learning", "nlp"]
```

Count-vector:

$$
[1, 1, 1, 0, 1, 1]
$$

## 4.6. Применение в NLP

- классификация документов;
- sentiment analysis;
- spam filtering;
- тематическая классификация;
- retrieval baseline.

## 4.7. Ограничения

Bag of Words:

- не учитывает порядок слов;
- не различает *not good* и *good* достаточно хорошо;
- даёт высокую размерность;
- создаёт сильную разреженность;
- не учитывает контекст;
- не моделирует синонимию и полисемию.

## 4.8. Исторический контекст

BoW был стандартом классического NLP и IR в течение многих лет. На нём строились поисковые системы, фильтры спама и ранние классификаторы документов.

## 4.9. Связь с другими методами

BoW — база для:

- TF;
- TF-IDF;
- Naive Bayes;
- Logistic Regression;
- LSA/SVD;
- retrieval по sparse vectors.

---

# 5. TF, TF-IDF и важность признаков

TF-IDF = Term Frequency * Inverse Document Frequency

## 5.1. Интуитивная идея

Не все слова одинаково полезны. Если слово встречается в документе часто, оно, возможно, важно для этого документа. Но если оно встречается почти везде, его различающая сила мала.

Поэтому нужен вес, который одновременно:

- усиливает локально важные слова;
- ослабляет глобально тривиальные слова.

Таким весом является TF-IDF.

То есть, слово важно, если оно:
- часто встречается в документе (высокий TF)
- редко встречается в корпусе (высокий IDF)

## 5.2. Формальная постановка

### Term Frequency

Показывает, насколько слово важно внутри конкретного документа

$$
\text{tf}(t,d) = \text{count}(t,d)
$$
или нормализованный вариант:
$$
\text{tf}(t,d) = \frac{\text{count}(t,d)}{\sum_{t'} \text{count}(t',d)}
$$

### Inverse Document Frequency

Показывает, насколько слово редкое во всём корпусе

$$
\text{idf}(t) = \log \frac{N}{df(t)}
$$
или со сглаживанием:
$$
\text{idf}(t) = \log \frac{N+1}{df(t)+1} + 1
$$

### TF-IDF
$$
\text{tfidf}(t,d) = \text{tf}(t,d)\cdot \text{idf}(t)
$$

## 5.3. Алгоритм

1. Посчитать частоты слов в каждом документе.
2. Посчитать document frequency $df(t)$.
3. Вычислить IDF.
4. Для каждой пары $(t,d)$ вычислить TF-IDF.
5. По желанию нормализовать векторы.

## 5.4. Псевдокод

```text
for term in vocabulary:
    df[term] = number_of_documents_containing(term)

for document in corpus:
    for term in document:
        tfidf[document, term] = tf(term, document) * log(N / df[term])
```

## 5.5. Пример

Если слово *nlp* встречается 5 раз в одном документе, но лишь в 10 документах из 10 000, его TF-IDF будет высоким. Если слово *the* встречается 20 раз, но есть почти в каждом документе, его TF-IDF будет низким.

## 5.6. Применение

- информационный поиск;
- ранжирование документов;
- baseline-классификация;
- кластеризация документов.

## 5.7. Ограничения

- порядок слов всё ещё игнорируется;
- семантика не моделируется;
- IDF плохо работает на очень маленьких корпусах;
- слова с одинаковым смыслом остаются разными измерениями.

## 5.8. Исторический контекст

TF-IDF — один из самых устойчивых и практически полезных методов в IR. Даже в эпоху dense retrieval его часто используют как baseline.

## 5.9. Связь с другими методами

TF-IDF естественно сочетается с:

- Logistic Regression;
- linear SVM;
- cosine similarity;
- nearest neighbors;
- LSA.

## 5.10 Интуиция значений

Важно: у TF-IDF нет фиксированной шкалы. Значения зависят от размера корпуса, способа нормализации, реализации (sklearn, raw, log-scaling и т.д.). К примеру sklearn нормализует в диапазон 0-1. Поэтому нельзя сравнивать значения между разными корпусами, и всегда нужно оценивать TF-IDF относительно других TF-IDF внутри документа. Но есть практические ориентиры:
- 0 – 0.1 - частые, слабоинформативные слова
- 0.1 – 0.5 - обычные слова, умеренная значимость, часто встречаются в теме, но не уникальны
- 0.5 – 1.0+ - редкие, специфичные слова, ключевые термины документа

---

# 6. Vector Space Models

## 6.1. Интуитивная идея

После векторизации текст становится точкой в многомерном пространстве. Тогда можно говорить не только о классах, но и о **геометрии смысла**:

- похожие документы должны быть близко;
- непохожие — далеко;
- направления и подпространства могут соответствовать темам и латентным факторам.

Курс вводит vector space models как способ представлять слова и документы в виде векторов и сравнивать их через similarity. Также в субтитрах подчёркивается, что близость вектора может отражать семантическую близость и использоваться в paraphrasing, QA и summarization.

## 6.2. Формальная постановка

Пусть документ или слово представлены вектором:

$$
x \in \mathbb{R}^d
$$

Тогда сходство можно оценивать разными метриками.

### Euclidean distance
$$
d(x,y)=\sqrt{\sum_{i=1}^{d}(x_i-y_i)^2}
$$

### Cosine similarity
$$
\cos(x,y)=\frac{x \cdot y}{\|x\|\|y\|}
$$

В текстовых задачах cosine similarity обычно лучше, потому что она чувствительна к направлению вектора, а не к его масштабу.

## 6.3. Алгоритм

1. Построить векторы документов.
2. Нормализовать их при необходимости.
3. Для любой пары документов посчитать cosine similarity.
4. Использовать similarity для retrieval, clustering или k-NN.

## 6.4. Псевдокод

```text
function cosine_similarity(x, y):
    return dot(x, y) / (norm(x) * norm(y))
```

## 6.5. Пример

Документы:

- D1: “learn nlp with python”
- D2: “study natural language processing using python”
- D3: “best pizza in town”

После TF-IDF и нормализации D1 и D2 будут ближе друг к другу, чем к D3.

## 6.6. Применение

- semantic-ish search в классическом приближении;
- поиск похожих документов;
- deduplication;
- recommendation;
- clustering.

## 6.7. Ограничения

Классические vector spaces всё ещё зависят от явного совпадения слов. Они не умеют по-настоящему захватывать контекст и синонимию так, как современные embeddings.

## 6.8. Исторический контекст

Vector Space Model — основа классического information retrieval. Отсюда выросли и dense embeddings: идея осталась прежней, но векторы стали обучаемыми и более семантическими.

## 6.9. Связь с другими методами

Vector space reasoning ведёт к:

- k-NN;
- ANN search;
- embeddings;
- semantic search;
- retrieval systems.

---

# 7. Logistic Regression for NLP

## 7.1. Интуитивная идея

Logistic Regression — это линейный классификатор, который вычисляет вероятность принадлежности текста к классу. В курсе он подаётся как важный и удобный baseline: его легко обучать, он интерпретируем и хорошо подходит для sentiment analysis на твитах.

Идея простая:

- текст превращается в вектор признаков $x$;
- модель вычисляет линейный score $z = \theta^T x$;
- затем преобразует его в вероятность через sigmoid.

## 7.2. Формальная постановка

Пусть:

- $x \in \mathbb{R}^d$ — вектор признаков;
- $y \in \{0,1\}$ — метка класса;
- $\theta \in \mathbb{R}^d$ — параметры модели.

### Линейный score
$$
z = \theta^T x
$$

### Sigmoid
$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

### Вероятность положительного класса
$$
P(y=1|x)=\sigma(\theta^T x)
$$

### Правило решения
$$
\hat y=
\begin{cases}
1, & \sigma(\theta^T x)\ge 0.5 \\
0, & \text{иначе}
\end{cases}
$$

Курс прямо использует threshold 0.5 и описывает проверку $\sigma(\theta^T x)\ge 0.5$ для формирования вектора предсказаний.

## 7.3. Функция потерь

Для одного объекта:

$$
L(y,\hat p)= -y\log \hat p -(1-y)\log(1-\hat p)
$$

Для всей выборки:

$$
J(\theta)= -\frac{1}{m}\sum_{i=1}^{m}
\left[
y^{(i)}\log \sigma(\theta^T x^{(i)})
+ (1-y^{(i)})\log(1-\sigma(\theta^T x^{(i)}))
\right]
$$

Курс отдельно объясняет интуицию этой функции: когда предсказание близко к истинной метке, loss мал; когда модель уверенно ошибается, loss резко растёт.

## 7.4. Градиентный спуск

Градиент:

$$
\nabla_\theta J(\theta)=\frac{1}{m}X^T(\hat y - y)
$$

Обновление параметров:

$$
\theta := \theta - \alpha \nabla_\theta J(\theta)
$$

где $\alpha$ — learning rate.

Курс описывает обучение именно как повторяющееся обновление $\theta$ в направлении антиградиента cost function до минимума.

## 7.5. Алгоритм

1. Preprocess тексты.
2. Построить признаки.
3. Инициализировать $\theta$.
4. Посчитать $\hat p = \sigma(X\theta)$.
5. Посчитать градиент.
6. Обновить $\theta$.
7. Повторять до сходимости.
8. На валидации считать accuracy.

## 7.6. Псевдокод

```text
initialize theta
repeat until convergence:
    z = X @ theta
    p = sigmoid(z)
    grad = (1/m) * X.T @ (p - y)
    theta = theta - alpha * grad
```

## 7.7. Пример

Пусть у твита есть вектор:

$$
x = [1, 8, 11]
$$

где:

- 1 — bias;
- 8 — сумма positive frequencies;
- 11 — сумма negative frequencies.

Тогда модель вычисляет:

$$
z = \theta^T x
$$

и после sigmoid получает вероятность положительного класса. В курсе как раз показана компактная 3-мерная схема признаков для твитов: bias + сумма положительных частот слов + сумма отрицательных частот слов.

## 7.8. Оценка качества

### Accuracy
$$
\text{Accuracy}=\frac{1}{m}\sum_{i=1}^{m}\mathbb{1}[\hat y^{(i)} = y^{(i)}]
$$

Курс подробно разбирает, как из вероятностей получить бинарные предсказания и затем сравнить их с true labels на validation set.

Дополнительно в реальных задачах нужны:

- precision;
- recall;
- F1-score;
- confusion matrix.

## 7.9. Применение

- sentiment analysis;
- spam detection;
- topic classification;
- toxic comment classification;
- intent classification.

## 7.10. Ограничения

- линейная граница решения;
- сильная зависимость от признаков;
- плохо учитывает порядок слов;
- без feature engineering не видит сложный контекст.

## 7.11. Исторический контекст

До нейросетевого бума Logistic Regression была одним из основных baseline-методов в классификации текста и остаётся важной и сегодня: если даже современная модель не бьёт хороший logistic baseline, значит что-то не так с данными или постановкой.

## 7.12. Связь с другими методами

Logistic Regression — мост между “простым счётом слов” и обучаемыми нейросетевыми моделями:

Bag of Words → linear model → probabilistic output → gradient optimization.

---

# 8. Naive Bayes

## 8.1. Интуитивная идея

Naive Bayes отвечает на вопрос: **насколько вероятно, что этот текст принадлежит классу $c$, если мы увидели его слова?**

Он особенно хорош как быстрый baseline для текста. Курс подчёркивает, что Naive Bayes прост в обучении, интерпретации и часто работает surprisingly well, хотя опирается на сильное допущение независимости слов.

## 8.2. Формальная постановка

По правилу Байеса:

$$
P(c|d)=\frac{P(d|c)P(c)}{P(d)}
$$

Поскольку $P(d)$ одинаково для всех классов при сравнении, достаточно максимизировать:

$$
P(d|c)P(c)
$$

### Наивное допущение
Если документ состоит из слов $w_1,\dots,w_n$, то:

$$
P(d|c)=\prod_{i=1}^{n} P(w_i|c)
$$

И тогда:

$$
P(c|d)\propto P(c)\prod_{i=1}^{n}P(w_i|c)
$$

## 8.3. Оценка вероятностей

### Prior
$$
P(c)=\frac{N_c}{N}
$$

### Likelihood
$$
P(w|c)=\frac{\text{count}(w,c)}{\sum_{w'}\text{count}(w',c)}
$$

### Laplace smoothing
$$
P(w|c)=\frac{\text{count}(w,c)+1}{\sum_{w'}\text{count}(w',c)+|V|}
$$

Курс подробно показывает проблему нулевой вероятности и вводит Laplacian smoothing как добавление 1 в числитель и $|V|$ в знаменатель.

## 8.4. Логарифмическая форма

Чтобы избежать numerical underflow, переходят к логарифмам:

$$
\log P(c|d) = \log P(c) + \sum_{i=1}^{n}\log P(w_i|c) + \text{const}
$$

Для бинарной классификации удобно рассматривать log-ratio:

$$
\log \frac{P(c=1|d)}{P(c=0|d)}
=
\log \frac{P(c=1)}{P(c=0)}
+
\sum_{i=1}^{n}\log \frac{P(w_i|c=1)}{P(w_i|c=0)}
$$

Курс вводит величину $\lambda(w)$ как логарифм отношения вероятностей слова в положительном и отрицательном классах и затем суммирует такие $\lambda$ по словам твита.

## 8.5. Алгоритм

1. Разделить корпус по классам.
2. Выполнить preprocessing.
3. Посчитать count(word, class).
4. Посчитать priors.
5. Посчитать smoothed likelihoods.
6. Посчитать log-likelihood ratios.
7. Для нового текста суммировать log prior и вклады слов.
8. Выбрать класс по знаку или максимуму score.

## 8.6. Псевдокод

```text
train:
    for each class c:
        count words in documents of class c
        compute P(c)
        compute P(w|c) with Laplace smoothing

predict(document):
    score[c] = log P(c)
    for word in preprocess(document):
        if word in vocabulary:
            score[c] += log P(word|c)
    return argmax_c score[c]
```

## 8.7. Пример

Если слово *happy* чаще встречается в positive tweets, чем в negative, его вклад в положительный класс будет положительным. Если *sad* чаще в negative, вклад будет отрицательным. Нейтральные слова почти сокращаются.

Именно так курс вводит “power words”: слова, которые существенно отличаются по вероятности между классами и поэтому определяют решение модели.

## 8.8. Применение

- sentiment analysis;
- authorship attribution;
- spam filtering;
- document filtering;
- word sense disambiguation.

Курс перечисляет эти применения явно.

## 8.9. Ограничения

Главное ограничение — **условная независимость признаков**. В реальном языке слова зависимы, порядок важен, а отрицание и сарказм легко ломают модель.

Курс отдельно показывает:

- потерю смысла на этапе preprocessing;
- проблему negation;
- проблему word order;
- сарказм и irony как типичные failure cases.

## 8.10. Исторический контекст

Naive Bayes десятилетиями использовался в спам-фильтрах, поиске и классификации документов. Он очень прост, но концептуально важен: даёт мост от частот слов к вероятностному моделированию языка.

## 8.11. Связь с другими методами

Naive Bayes ведёт к более сложным probabilistic NLP methods:

- n-gram language models;
- HMM;
- CRF;
- нейронные языковые модели.

---

# 9. k-Nearest Neighbors и Similarity Search

## 9.1. Интуитивная идея

Если документы представлены векторами, то новый документ можно классифицировать по соседям: посмотреть, какие тексты рядом с ним в пространстве признаков.

## 9.2. Формальная постановка

Для нового вектора $x$ ищутся $k$ ближайших соседей по метрике $d(x, x_i)$. Класс выбирается по большинству или по взвешенному голосованию.

## 9.3. Алгоритм

1. Векторизовать все документы.
2. Для нового текста посчитать расстояние до обучающих примеров.
3. Взять $k$ ближайших.
4. Агрегировать их метки.

## 9.4. Ограничения

- дорогой inference на больших корпусах;
- чувствительность к масштабу признаков;
- в очень высокой размерности distance degrades.

## 9.5. Практическая роль

Для классического NLP важнее не сама классификация k-NN, а идея **nearest neighbors over vectors**, из которой вырастает retrieval, semantic search и позднее ANN/FAISS.

---

# 10. Information Retrieval в векторной постановке

## 10.1. Интуитивная идея

Информационный поиск в базовом виде — это ранжирование документов по запросу. Если и запрос, и документы представлены векторами, задача сводится к поиску ближайших по similarity.

## 10.2. Формальная постановка

Пусть запрос $q$ и документ $d_i$ — векторы. Тогда score документа:

$$
s_i = \cos(q, d_i)
$$

Документы сортируются по убыванию $s_i$.

## 10.3. Практические структуры

- inverted index;
- sparse retrieval;
- TF-IDF ranking;
- позже — ANN search.

## 10.4. Ограничения

- lexical mismatch;
- отсутствие настоящей семантики;
- плохая работа с перефразировками.

---

# 11. Dimensionality Reduction: PCA, SVD, LSA

## 11.1. Интуитивная идея

BoW и TF-IDF создают огромные разреженные пространства. Часть измерений шумовая, часть коррелирована, часть описывает одну и ту же скрытую тему. Методы снижения размерности пытаются сжать данные, сохранив максимум полезной информации.

В субтитрах PCA объясняется как проекция данных на новые некоррелированные направления, задаваемые собственными векторами covariance matrix; величина сохраняемой информации связана с eigenvalues.

## 11.2. Формальная постановка

Пусть $X\in\mathbb{R}^{n\times d}$ — матрица признаков.

### PCA
1. Центрировать данные.
2. Посчитать covariance matrix:
$$
\Sigma = \frac{1}{n}X^TX
$$
3. Найти собственные значения и собственные векторы:
$$
\Sigma u_i = \lambda_i u_i
$$
4. Взять первые $k$ eigenvectors:
$$
U_k = [u_1,\dots,u_k]
$$
5. Спроецировать данные:
$$
Z = XU_k
$$

## 11.3. SVD и LSA

Для матрицы document-term:

$$
X = U\Sigma V^T
$$

Если оставить только первые $k$ сингулярные значения:

$$
X_k = U_k \Sigma_k V_k^T
$$

то получаем приближение меньшей размерности. В NLP это лежит в основе **Latent Semantic Analysis**, где скрытые компоненты интерпретируются как латентные темы/факторы.

## 11.4. Алгоритм

1. Построить DTM или TF-IDF matrix.
2. Нормализовать при необходимости.
3. Применить PCA или SVD.
4. Выбрать $k$ компонент.
5. Использовать reduced vectors для визуализации, clustering или дальнейшего обучения.

## 11.5. Пример

Если документы про “spacecraft”, “satellite”, “orbit” и “launch vehicle” часто используют связанные слова, SVD может свернуть их в общее латентное измерение “space topic”, даже если слова не полностью совпадают.

## 11.6. Применение

- визуализация корпусов;
- ускорение обучения;
- удаление шума;
- latent topic structure;
- retrieval in reduced space.

## 11.7. Ограничения

- компоненты не всегда интерпретируемы;
- линейность метода;
- потеря части информации;
- на больших данных dense decompositions дорогие.

## 11.8. Исторический контекст

До нейронных embeddings LSA была одной из ключевых попыток выйти за рамки простого совпадения слов и извлечь латентную семантику из матриц совместной встречаемости.

## 11.9. Связь с другими методами

PCA/SVD/LSA подготавливают переход к embeddings: идея та же — представлять слова и документы в компактном пространстве, где геометрия отражает структуру данных.

---

# 12. Практический pipeline раздела

Итоговый pipeline для классического NLP выглядит так:

```text
raw text
→ preprocessing
→ vocabulary
→ BoW / TF-IDF / count features
→ classifier or similarity model
→ evaluation
```

### Для классификации
- Logistic Regression
- Naive Bayes
- k-NN

### Для поиска и близости
- cosine similarity
- nearest neighbors
- inverted index
- TF-IDF ranking

### Для анализа структуры
- PCA
- SVD
- LSA

---

# 13. Короткие Python-примеры

## 13.1. Preprocessing

```python
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", " ", text)   # remove URLs
    text = re.sub(r"@\w+", " ", text)             # remove handles
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.isalpha()]
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [stemmer.stem(t) for t in tokens]
    return tokens
```

## 13.2. Bag of Words / TF-IDF

```python
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts = [
    "I am happy because I am learning NLP",
    "I am sad because this task is hard",
]

bow = CountVectorizer()
X_bow = bow.fit_transform(texts)

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(texts)
```

## 13.3. Logistic Regression

```python
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

X_train = [
    "I love this movie",
    "This is terrible",
    "Amazing work",
    "I hate this",
]
y_train = [1, 0, 1, 0]

model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
pred = model.predict(["I love this work", "This is awful"])
print(pred)
```

## 13.4. Naive Bayes

```python
from sklearn.naive_bayes import MultinomialNB

nb_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", MultinomialNB())
])

nb_model.fit(X_train, y_train)
pred = nb_model.predict(["I love this work", "This is awful"])
print(pred)
```

## 13.5. Cosine Similarity

```python
from sklearn.metrics.pairwise import cosine_similarity

docs = [
    "natural language processing with python",
    "nlp course and machine learning",
    "pizza and pasta restaurant"
]

vec = TfidfVectorizer()
X = vec.fit_transform(docs)

sim_01 = cosine_similarity(X[0], X[1])[0, 0]
sim_02 = cosine_similarity(X[0], X[2])[0, 0]

print(sim_01, sim_02)
```

## 13.6. PCA

```python
from sklearn.decomposition import PCA
import numpy as np

X_dense = X.toarray()
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X_dense)

print(X_2d)
```

---

# 14. Что нужно усвоить по этому разделу

После изучения этого блока ты должен уметь:

1. объяснить, зачем нужен preprocessing и где он может навредить;
2. построить vocabulary и document-term matrix;
3. различать BoW, TF, TF-IDF;
4. объяснить разреженность и высокую размерность текстовых признаков;
5. вывести и интерпретировать logistic regression для текста;
6. объяснить Bayes rule, likelihood, prior, Laplace smoothing и log-likelihood;
7. использовать cosine similarity для сравнения текстов;
8. понимать, зачем нужны PCA/SVD/LSA;
9. видеть ограничения классического NLP и понимать, почему дальше появились embeddings.

---

# 15. Итог раздела

Этот раздел показывает первую зрелую форму NLP: язык представляется как набор признаков, а текст — как вектор в пространстве высокой размерности. На этих векторах уже можно:

- обучать классификаторы;
- считать вероятности;
- искать похожие документы;
- визуализировать данные;
- выделять латентную структуру.

Но одновременно здесь видны и пределы классического подхода:

- порядок слов почти теряется;
- отрицание и контекст обрабатываются плохо;
- семантика задаётся косвенно;
- признаки приходится конструировать вручную.

Именно из этих ограничений естественно вырастают следующие этапы эволюции NLP:

**Bag of Words → weighted vectors → probabilistic models → embeddings → sequence models → attention → transformers**.
